<a href="https://colab.research.google.com/github/srihari-g-acharya-ci109/ML2LAB/blob/main/ML4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# load mnist sample, with fallback if not running inside colab sample dir
mnist_path = "/content/sample_data/mnist_train_small.csv"
if os.path.exists(mnist_path):
    raw_df = pd.read_csv(mnist_path, header=None)
else:
    # generate a small synthetic dataframe if mnist is missing locally
    import numpy as np
    np.random.seed(42)
    fake_labels = np.random.choice(range(10), size=100)
    fake_pixels = np.random.randint(0, 255, size=(100, 784))
    raw_df = pd.DataFrame(np.column_stack([fake_labels, fake_pixels]))

print("Dataset shape:", raw_df.shape)

# treat digit 0 as positive, others as negative
raw_df['Target_Digit'] = raw_df[0]
raw_df["Class"] = raw_df["Target_Digit"].apply(lambda x: "Positive" if x == 0 else "Negative")

# pick 6 pixel indices as binary attributes
pixel_indices = [100, 150, 200, 250, 300, 350]
features = [f'pixel_{i}' for i in pixel_indices]

for i, col_idx in enumerate(pixel_indices):
    raw_df[features[i]] = raw_df[col_idx].apply(lambda x: 1 if x > 0 else 0)

print("Class counts:
", raw_df["Class"].value_counts())

# sample 8 positive and 7 negative instances
pos = raw_df[raw_df["Class"] == "Positive"].sample(n=min(8, len(raw_df[raw_df["Class"] == "Positive"])), random_state=42)
neg = raw_df[raw_df["Class"] == "Negative"].sample(n=min(7, len(raw_df[raw_df["Class"] == "Negative"])), random_state=42)

sample_df = pd.concat([pos, neg]).sample(frac=1, random_state=42).reset_index(drop=True)
print("
Selected examples for Candidate-Elimination:")
display(sample_df[['Target_Digit', 'Class'] + features])

In [ ]:
def covers(h, x):
    for hi, xi in zip(h, x):
        if hi == "Phi":
            return False
        if hi != "?" and hi != xi:
            return False
    return True

def more_general(h1, h2):
    for a, b in zip(h1, h2):
        if b == "Phi":
            continue
        if a == "Phi":
            return False
        if b != "?" and a == "?":
            continue
        if a != "?" and a != b:
            return False
    return True

def get_specializations(g, example, domains):
    specs = set()
    for i in range(len(g)):
        if g[i] == "?":
            for val in domains[features[i]]:
                if val != example[i]:
                    new_g = list(g)
                    new_g[i] = val
                    specs.add(tuple(new_g))
    return specs

domains = {f: [0, 1] for f in features}
S = {tuple(["Phi"] * len(features))}
G = {tuple(["?"] * len(features))}

S_history = []
G_history = []

for _, row in sample_df.iterrows():
    example = tuple(row[f] for f in features)
    label = row["Class"]

    if label == "Positive":
        # remove from G anything inconsistent with positive
        G = {g for g in G if covers(g, example)}
        new_S = set()
        for s in S:
            if covers(s, example):
                new_S.add(s)
            else:
                h = list(s)
                for i in range(len(features)):
                    if s[i] == "Phi":
                        h[i] = example[i]
                    elif s[i] != example[i]:
                        h[i] = "?"
                h_t = tuple(h)
                if any(more_general(g, h_t) for g in G):
                    new_S.add(h_t)
        S = new_S

    else:
        # negative: prune S
        S = {s for s in S if not covers(s, example)}
        new_G = set()
        for g in G:
            if not covers(g, example):
                new_G.add(g)
            else:
                specs = get_specializations(g, example, domains)
                for sp in specs:
                    if any(more_general(sp, s) for s in S):
                        new_G.add(sp)
        G = new_G

    S_history.append(S.copy())
    G_history.append(G.copy())

print("Final S boundary:")
for s in S:
    print(" ", dict(zip(features, s)))

print("
Final G boundary:")
for g in G:
    print(" ", dict(zip(features, g)))

# plot boundary sizes across steps
s_counts = [len(s) for s in S_history]
g_counts = [len(g) for g in G_history]
steps = range(1, len(sample_df) + 1)

plt.figure(figsize=(9, 4))
plt.plot(steps, s_counts, marker="o", label="S Size")
plt.plot(steps, g_counts, marker="s", label="G Size")
plt.xlabel("Example Step")
plt.ylabel("Number of Hypotheses")
plt.title("Candidate-Elimination: S & G Boundary Size Evolution")
plt.xticks(list(steps))
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()